In [1]:
import numpy as np
import re
from sklearn.decomposition import PCA
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer, util

/home/koki/anaconda3/envs/seal3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-11 16:16:25.308581: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-11 16:16:25.309462: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-11 16:16:25.311412: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-11 16:16:25.317110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to re

In [3]:
def split_entity(s, chars_to_remove=['_', '.', ',', '(', ')', '[', ']', '!']):
    t = re.sub( r"([A-Z]|_)", r" \1", s).split()
    res = ' '.join(t)
    sc = set(chars_to_remove)
    return ''.join([c for c in res if c not in sc])

In [6]:
np.multiply([1,2,3], [3,4,5])

array([ 3,  8, 15])

In [163]:
datapath = 'data/fr_en/'

def read_entities_map(datapath, filename):
    uri_map = {}
    f = open(datapath+filename, 'r')
    for line in f:
        nr_id, uri = line.split()[0], line.split()[1]
        uri_map[nr_id] = uri
    f.close()
    return uri_map

ent_map1 = read_entities_map(datapath, 'ent_ids_1')
ent_map2 = read_entities_map(datapath, 'ent_ids_2')

rel_map1 = read_entities_map(datapath, 'rel_ids_1')
rel_map2 = read_entities_map(datapath, 'rel_ids_2')

In [164]:
ent_map1['2786']

'http://fr.dbpedia.org/resource/Rodrigo_Rato'

In [165]:
def get_graph(datapath, triplesfile, ent_map, rel_map):
    graph = {}
    f = open(datapath+triplesfile, 'r')
    for line in f:
        ls = line.split()
        h, r, t = ent_map[ls[0]].split('/')[-1], rel_map[ls[1]].split('/')[-1], ent_map[ls[2]].split('/')[-1]
        graph.setdefault(h, [])
        graph[h].append((r, t))
        graph.setdefault(t, [])
        graph[t].append(('reverse_'+r, h))
    f.close()
    return graph

G1 = get_graph(datapath, 'triples_1', ent_map1, rel_map1)
G2 = get_graph(datapath, 'triples_2', ent_map2, rel_map2)

In [167]:
len(G1), len(G2)

(19660, 19993)

In [75]:
def random_walk(G, start_node, nr_walks, walk_length, seed=73):
    walks = []
    np.random.seed(seed)
    for _ in range(nr_walks):
        neighbors = G[start_node]
        walk = [start_node]
        for _ in range(walk_length):
            idx = np.random.randint(len(neighbors))
            walk.append(neighbors[idx][0])
            walk.append(neighbors[idx][1])
            #print(neighbors[idx])
            neighbors = G[neighbors[idx][1]]
        walks.append(walk)
    return walks         

In [168]:
corpus = []
for node in G1.keys():
    corpus.extend(random_walk(G1, node, nr_walks=5, walk_length=10))

In [79]:
len(corpus), corpus[:2]

(98300,
 [['Rodrigo_Rato',
   'reverse_prédécesseur',
   'José_Montilla',
   'coalition',
   'Initiative_pour_la_Catalogne_Verts',
   'nomsMembresAssemblée',
   'Parlement_de_Catalogne',
   'reverse_mandant',
   'Président_de_la_Généralité_de_Catalogne',
   'reverse_fonction',
   'Artur_Mas',
   'parti',
   'Convergence_démocratique_de_Catalogne',
   'assemblée1groupe',
   'Sénat_(Espagne)',
   'reverse_assemblée2groupe',
   'Parti_populaire_(Espagne)',
   'reverse_parti',
   'Malaga',
   'parti',
   'Parti_populaire_(Espagne)'],
  ['Rodrigo_Rato',
   'université',
   'Université_complutense_de_Madrid',
   'reverse_université',
   'José_María_Aznar',
   'reverse_successeur',
   'Manuel_Fraga',
   'fonction',
   'Alliance_populaire_(Espagne)',
   'fusionnéDans',
   'Parti_populaire_(Espagne)',
   'assemblée2groupe',
   'Sénat_(Espagne)',
   'reverse_fonction',
   'Esperanza_Aguirre',
   'reverse_successeur',
   'Alberto_Ruiz-Gallardón',
   'successeur',
   'Ángel_Acebes',
   'prédécesse

In [131]:
def split_entity(s, chars_to_remove=['_', '.', ',', '(', ')', '[', ']', '!']):
    t = re.sub( r"([A-Z]|_)", r" \1", s).split()
    res = ' '.join(t)
    sc = set(chars_to_remove)
    return ''.join([c for c in res if c not in sc])
  

split_entity('Antonio_López_de_Santa_Anna')

'Antonio  López de  Santa  Anna'

In [137]:
llm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [138]:
embeddings = llm_model.encode(["Hello World."])
embeddings

array([[ 1.07261397e-01,  2.34287977e-01,  3.14048678e-01,
         2.62109041e-01, -2.16763735e-01, -4.55232620e-01,
         4.57824618e-01,  1.29924908e-01, -2.19045117e-01,
         3.85543495e-01,  1.67922422e-01, -6.13745391e-01,
        -1.55247241e-01,  5.52761694e-03,  2.70280868e-01,
         1.83856577e-01,  1.54833823e-01, -2.02819392e-01,
        -5.51075876e-01, -2.71568835e-01,  1.26062587e-01,
         1.17023345e-02, -2.61322081e-01,  2.10972384e-01,
         1.79251522e-01, -1.46688591e-03,  5.47117218e-02,
         4.51085657e-01,  8.80281031e-02, -2.29639173e-01,
        -2.65739202e-01, -1.28677994e-01,  4.57941234e-01,
         5.35214618e-02, -2.31479555e-01,  4.47780788e-01,
        -4.45918143e-02, -9.41646546e-02, -1.03203431e-01,
         1.08495817e-01,  2.27830887e-01, -4.98536415e-02,
         2.17972130e-01,  2.15701699e-01,  4.31880951e-02,
        -2.73072898e-01, -1.91237349e-02,  6.93062395e-02,
         8.75901729e-02,  5.28885946e-02,  1.38922736e-0

In [139]:
embeddings = llm_model.encode(["Hello     World.", "Hallo Welt", "Hola mundo", "Dog"])
llm_model.similarity(embeddings, embeddings)

tensor([[1.0000, 0.9370, 0.8715, 0.2625],
        [0.9370, 1.0000, 0.9680, 0.3400],
        [0.8715, 0.9680, 1.0000, 0.3263],
        [0.2625, 0.3400, 0.3263, 1.0000]])

In [136]:
model2 = Word2Vec(vector_size=1, window=5, min_count=1, workers=1)
model2.build_vocab(corpus)
len(sorted(model2.wv.index_to_key))

21360

In [150]:
def get_LLM_embeddings(llm_model, corpus, emb_dim):
    model_w2v = Word2Vec(vector_size=1, window=5, min_count=1, workers=1)
    model_w2v.build_vocab(corpus)
    vocabulary = sorted(model_w2v.wv.index_to_key)
    sentences = [split_entity(ent) for ent in vocabulary]
    print('Sentences computed')
    embeddings = llm_model.encode(sentences)
    pca = PCA(n_components=emb_dim)
    reduced_dim_embeddings = pca.fit_transform(embeddings)
    assert len(vocabulary) == reduced_dim_embeddings.shape[0]
    print('PCA dim reduction finished')
    emb_dic = {ent: emb for ent, emb in zip(vocabulary, reduced_dim_embeddings)}
    return emb_dic

initial_embeddings = get_LLM_embeddings(llm_model, corpus, emb_dim=64)


Sentences computed
PCA dim reduction finished


In [154]:
def write_to_w2v_format(path, fname, embeddings_map):
    f = open(path + fname, 'w')
    f.write(str(len(embeddings_map)) + ' ' + str(len(list(embeddings_map.values())[0])) + '\n')
    for ent, emb in embeddings_map.items():
        f.write(ent + ' ')
        for val in emb:
            f.write(str(val) + ' ')
        f.write('\n')
    f.close()

write_to_w2v_format('data/embeddings/', 'llm_embeddings.txt', initial_embeddings)

In [161]:
model_w2v = Word2Vec(vector_size=64, window=5, min_count=1, workers=1)
model_w2v.build_vocab(corpus)
model_w2v.wv.vectors_lockf = np.ones(len(model_w2v.wv), dtype=np.float32)
model_w2v.wv.intersect_word2vec_format("data/embeddings/llm_embeddings.txt")
model_w2v.wv.vectors_lockf = np.ones(len(model_w2v.wv), dtype=np.float32)
model_w2v.train(corpus, total_examples=3, epochs=5)
model_w2v.wv.save_word2vec_format("data/embeddings/word2vec_embeddings.txt")

In [159]:
len(model_w2v.wv.index_to_key), len(initial_embeddings)

(21360, 21360)